# 03 — Demand Estimation (transparent baseline)

**目标**：在 Step 2 (clean) + Step 3 (EDA freeze) 的地基上，给出第一版可解释的需求模型。**先跑透明 OLS with high-dim FE，不做 IV**；IV / cost-shock / Hausman 放到后续 robustness notebook 里。

**主模型**：
$$\log(q_{ist}) = \alpha_{is} + \gamma_t + \beta_\text{own}\,\log(p_{ist}) + \beta_\text{cross}\,\log(p^{comp}_{ist}) + \theta\,\text{promo\_any}_{ist} + \varepsilon_{ist}$$

- $i$ = brand × size，$s$ = store，$t$ = week；$\alpha_{is}$ = brand-size-store FE，$\gamma_t$ = week FE
- **主面板**：`data/processed/brand_size_store_week_panel.parquet` (2.7M rows)
- **Robustness**：`data/processed/sku_store_week_panel.parquet` (4.6M rows), FE = UPC×store + week
- **工具**：`linearmodels.iv.absorbing.AbsorbingLS`（高维 FE 吸收，避免显式 dummy 爆内存）

**产出**：
1. `reports/demand_model_summary.md` —— 系数、SE、R²、smearing、holdout 指标
2. `reports/figures/demand_*.png` —— 系数图、holdout 预测散点
3. `data/processed/model_coefficients.csv` —— 系数表，memo/app 直接读
4. `data/processed/demand_modeling_dataset.parquet` —— 进入 fit 的清洗后数据


In [ ]:
from __future__ import annotations
from pathlib import Path
from datetime import datetime
import sys, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

PROCESSED = PROJECT_ROOT / 'data' / 'processed'
REPORTS   = PROJECT_ROOT / 'reports'
FIGURES   = REPORTS / 'figures'
for p in (PROCESSED, FIGURES): p.mkdir(parents=True, exist_ok=True)

summary_lines: list[str] = []
def log(msg: str) -> None:
    print(msg); summary_lines.append(msg)

log('# Demand Model Summary — baseline')
log(f'Generated: {datetime.now().isoformat(timespec="seconds")}')
log('')


## 1. Load panels


In [ ]:
bs = pd.read_parquet(PROCESSED / 'brand_size_store_week_panel.parquet')
upc = pd.read_parquet(PROCESSED / 'sku_store_week_panel.parquet')

log('## 1. Load')
log(f'- baseline panel (brand × size × store × week): {len(bs):,} rows, '
    f'{bs["brand_final"].nunique()} brands, {bs["STORE"].nunique()} stores, '
    f'{bs["WEEK"].nunique()} weeks')
log(f'- robustness panel (UPC × store × week): {len(upc):,} rows')


## 2. Construct modeling fields

- `log_q = log(quantity)`、`log_p = log(unit_price)`
- 过滤 `quantity > 0` 且 `unit_price > 0`（聚合已保证，但稳妥起见）
- 同时剔除 `cell_quality_flag` 含 `low_quantity` 的 cell（MOVE<3，噪声过大）


In [ ]:
# Baseline panel modeling dataset
mdl = bs.loc[(bs['quantity'] > 0) & (bs['unit_price'] > 0)].copy()
mdl = mdl.loc[~mdl['cell_quality_flag'].fillna('').str.contains('low_quantity')]
mdl['log_q']  = np.log(mdl['quantity'].astype('float64'))
mdl['log_p']  = np.log(mdl['unit_price'].astype('float64'))
mdl['promo_any_int'] = mdl['promo_any'].astype('int8')

log('## 2. Modeling dataset (baseline)')
log(f'- rows: {len(mdl):,} (dropped {len(bs)-len(mdl):,} for low-quantity/zero)')
log(f'- promo_any share: {mdl["promo_any_int"].mean()*100:.2f}%')
log(f'- log_q: mean={mdl["log_q"].mean():.2f}, sd={mdl["log_q"].std():.2f}')
log(f'- log_p: mean={mdl["log_p"].mean():.2f}, sd={mdl["log_p"].std():.2f}')


## 3. Competitor price index

Within each `(STORE, WEEK)`, build a quantity-weighted mean `unit_price` of **other brands** (different `brand_final`). This is the Hausman-style cross-brand competitor proxy for shelf substitution.

$$p^{comp}_{ist} = \frac{\sum_{j \neq b(i)} p_{jst}\, q_{jst}}{\sum_{j \neq b(i)} q_{jst}}$$


In [ ]:
# vectorized: store-week totals minus within-brand sums
mdl['price_move'] = mdl['unit_price'] * mdl['quantity']

sw_totals = mdl.groupby(['STORE','WEEK'], observed=True).agg(
    sw_price_move=('price_move', 'sum'),
    sw_q=('quantity', 'sum'),
).reset_index()

bsw_totals = mdl.groupby(['STORE','WEEK','brand_final'], observed=True).agg(
    bsw_price_move=('price_move', 'sum'),
    bsw_q=('quantity', 'sum'),
).reset_index()

mdl = mdl.merge(sw_totals,  on=['STORE','WEEK'],                how='left')
mdl = mdl.merge(bsw_totals, on=['STORE','WEEK','brand_final'],  how='left')

mdl['comp_price_move'] = mdl['sw_price_move'] - mdl['bsw_price_move']
mdl['comp_q']          = mdl['sw_q']          - mdl['bsw_q']

# rows where focal brand is the only brand in that store-week → comp_q = 0
valid = mdl['comp_q'] > 0
mdl.loc[valid, 'comp_price'] = mdl.loc[valid, 'comp_price_move'] / mdl.loc[valid, 'comp_q']
mdl['log_comp_price'] = np.log(mdl['comp_price'].astype('float64'))

n_drop = (~valid).sum()
mdl = mdl.loc[valid].copy()

log('## 3. Competitor price index')
log(f'- dropped rows with no competitor in same store-week: {n_drop:,}')
log(f'- remaining rows: {len(mdl):,}')
log(f'- log_comp_price: mean={mdl["log_comp_price"].mean():.2f}, sd={mdl["log_comp_price"].std():.2f}')

# brand-size-store key for FE absorption
mdl['brand_size_store'] = (
    mdl['brand_final'].astype('string') + '|' +
    mdl['size_oz_rounded'].round(2).astype('string') + '|' +
    mdl['STORE'].astype('string')
).astype('category')
mdl['WEEK_cat'] = mdl['WEEK'].astype('category')

log(f'- brand-size-store units: {mdl["brand_size_store"].nunique():,}')
log(f'- week units: {mdl["WEEK_cat"].nunique()}')


## 4. Baseline FE model: own price + promo

$\log q = \alpha_{is} + \gamma_t + \beta_\text{own} \log p + \theta\,\text{promo\_any} + \varepsilon$

Using `linearmodels.iv.absorbing.AbsorbingLS` to absorb `brand_size_store` (~18K) and `WEEK_cat` (366) without materializing dummies.


In [ ]:
from linearmodels.iv.absorbing import AbsorbingLS

def fit_absorbing(data: pd.DataFrame, y: str, X: list[str], absorb_cols: list[str]):
    """Fit AbsorbingLS with cluster-robust SE at the first absorbed FE."""
    dep = data[y].astype('float64')
    exog = data[X].astype('float64').assign(const=1.0)[['const', *X]]
    absorb = data[absorb_cols]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', FutureWarning)
        warnings.simplefilter('ignore', RuntimeWarning)
        mod = AbsorbingLS(dep, exog, absorb=absorb)
        res = mod.fit(cov_type='clustered', clusters=data[absorb_cols[0]])
    return res

res1 = fit_absorbing(
    mdl, y='log_q',
    X=['log_p', 'promo_any_int'],
    absorb_cols=['brand_size_store', 'WEEK_cat'],
)

log('## 4. Baseline FE model (own price + promo)')
log(f'- n obs = {int(res1.nobs):,}')
log(f'- R² within absorbed = {res1.rsquared:.4f}')
log(f'- β_own  (log_p)         = {res1.params["log_p"]:+.4f}  [SE {res1.std_errors["log_p"]:.4f}]')
log(f'- θ (promo_any)          = {res1.params["promo_any_int"]:+.4f}  [SE {res1.std_errors["promo_any_int"]:.4f}]')


## 5. Add cross-price term


In [ ]:
res2 = fit_absorbing(
    mdl, y='log_q',
    X=['log_p', 'log_comp_price', 'promo_any_int'],
    absorb_cols=['brand_size_store', 'WEEK_cat'],
)

log('## 5. FE model with cross price')
log(f'- n obs = {int(res2.nobs):,}')
log(f'- R² within absorbed = {res2.rsquared:.4f}')
for k in ['log_p', 'log_comp_price', 'promo_any_int']:
    log(f'- β[{k}] = {res2.params[k]:+.4f}  [SE {res2.std_errors[k]:.4f}]')


## 6. Robustness: UPC-level model

Same specification, on the UPC-store-week panel. UPC×store as absorbed unit, week FE as before. This is slower (4.6M rows, ~36K pair FEs) but uses the same `linearmodels` code path.


In [ ]:
# build UPC-level modeling frame analogous to §2 + §3
umdl = upc.loc[(upc['MOVE'] > 0) & (upc['unit_price'] > 0)].copy()
umdl['log_q'] = np.log(umdl['MOVE'].astype('float64'))
umdl['log_p'] = np.log(umdl['unit_price'].astype('float64'))
umdl['promo_any_int'] = umdl['promo'].astype('int8')

# attach brand from brand_mapping to build UPC competitor price
bmap = pd.read_csv(PROCESSED / 'brand_mapping.csv', usecols=['UPC','brand_final'])
umdl['UPC_int'] = umdl['UPC'].astype('int64')
umdl = umdl.merge(bmap, left_on='UPC_int', right_on='UPC', how='left', suffixes=('','_meta'))
umdl = umdl.drop(columns=[c for c in ['UPC_meta'] if c in umdl.columns])

umdl = umdl.loc[umdl['brand_final'].notna() & (umdl['brand_final'] != 'Unknown')].copy()
umdl['price_move'] = umdl['unit_price'] * umdl['MOVE']

sw = umdl.groupby(['STORE','WEEK'], observed=True).agg(
    sw_price_move=('price_move','sum'), sw_q=('MOVE','sum')).reset_index()
bsw = umdl.groupby(['STORE','WEEK','brand_final'], observed=True).agg(
    bsw_price_move=('price_move','sum'), bsw_q=('MOVE','sum')).reset_index()

umdl = umdl.merge(sw,  on=['STORE','WEEK'], how='left')
umdl = umdl.merge(bsw, on=['STORE','WEEK','brand_final'], how='left')
umdl['comp_q'] = umdl['sw_q'] - umdl['bsw_q']
umdl = umdl.loc[umdl['comp_q'] > 0].copy()
umdl['comp_price'] = (umdl['sw_price_move'] - umdl['bsw_price_move']) / umdl['comp_q']
umdl['log_comp_price'] = np.log(umdl['comp_price'])

umdl['upc_store'] = (umdl['UPC_int'].astype('string') + '|' + umdl['STORE'].astype('string')).astype('category')
umdl['WEEK_cat']  = umdl['WEEK'].astype('category')

res_upc = fit_absorbing(
    umdl.rename(columns={'upc_store': 'brand_size_store'}),  # reuse helper; cluster label is the absorbed id
    y='log_q',
    X=['log_p', 'log_comp_price', 'promo_any_int'],
    absorb_cols=['brand_size_store', 'WEEK_cat'],
)

log('## 6. Robustness UPC-level model')
log(f'- n obs = {int(res_upc.nobs):,}')
log(f'- R² within absorbed = {res_upc.rsquared:.4f}')
for k in ['log_p', 'log_comp_price', 'promo_any_int']:
    log(f'- β[{k}] = {res_upc.params[k]:+.4f}  [SE {res_upc.std_errors[k]:.4f}]')


## 7. Duan smearing factor

Log-linear models produce biased point forecasts after retransformation $\exp(\widehat{\log q})$. Correction:

$$\widehat{S} = \frac{1}{N}\sum_n \exp(\widehat{\varepsilon}_n)$$

Apply to predictions via $\widehat{q} = \widehat{S}\cdot\exp(\widehat{\log q})$.


In [ ]:
resid = res2.resids.astype('float64')
smearing = float(np.exp(resid).mean())
smearing_upc = float(np.exp(res_upc.resids.astype('float64')).mean())

log('## 7. Smearing')
log(f'- baseline smearing factor S = {smearing:.4f}')
log(f'- UPC robustness smearing S = {smearing_upc:.4f}')
log(f'  (S≈1 indicates near-symmetric residuals; >1 means naive exp undershoots mean)')


## 8. Holdout prediction sanity check

Hold out **last 20 weeks** (roughly a quarter). Re-fit baseline model on train; predict on holdout with smearing; report MAPE on cell-level `quantity`.


In [ ]:
holdout_cutoff = mdl['WEEK'].max() - 20
train = mdl.loc[mdl['WEEK'] <= holdout_cutoff].copy()
test  = mdl.loc[mdl['WEEK'] >  holdout_cutoff].copy()
# retain only test rows whose brand_size_store appears in train (FE needs to be identifiable)
test = test.loc[test['brand_size_store'].isin(train['brand_size_store'].unique())]
test = test.loc[test['WEEK_cat'].isin(train['WEEK_cat'].unique()) == False]  # unseen weeks → can't predict
# ^ the above keeps only rows from *unseen* weeks, which is what holdout requires;
# however, unseen weeks aren't in train's WEEK FE, so we project using mean week FE = 0 under within transform.
# Simpler approach: predict using train-estimated coefficients but use the *holdout* week's computed log_comp_price
# (which only depends on current prices, not on the FE estimates).

res_tr = fit_absorbing(train, y='log_q',
                       X=['log_p','log_comp_price','promo_any_int'],
                       absorb_cols=['brand_size_store','WEEK_cat'])
S_tr = float(np.exp(res_tr.resids.astype('float64')).mean())

# recover FE contributions row-wise: fitted_values (includes FE) − linear predictor from coefs·X
coefs = res_tr.params
fitted_vals = np.asarray(res_tr.fitted_values).ravel()
linear_pred_train = (coefs['const']
    + coefs['log_p']          * train['log_p'].to_numpy()
    + coefs['log_comp_price'] * train['log_comp_price'].to_numpy()
    + coefs['promo_any_int']  * train['promo_any_int'].to_numpy())
fe_contrib = pd.Series(fitted_vals - linear_pred_train, index=train.index)

bss_fe = fe_contrib.groupby(train['brand_size_store'], observed=True).mean()
wk_fe  = (fe_contrib - train['brand_size_store'].map(bss_fe)).groupby(
            train['WEEK_cat'], observed=True).mean()

test_pred_linear = (coefs['const']
                    + coefs['log_p'] * test['log_p']
                    + coefs['log_comp_price'] * test['log_comp_price']
                    + coefs['promo_any_int'] * test['promo_any_int']
                    + test['brand_size_store'].map(bss_fe).fillna(bss_fe.mean())
                    + test['WEEK_cat'].map(wk_fe).fillna(0.0))

test['pred_q'] = S_tr * np.exp(test_pred_linear)
test['q_actual'] = test['quantity']
test = test.loc[test['pred_q'].notna() & np.isfinite(test['pred_q'])].copy()

mape = (np.abs(test['pred_q'] - test['q_actual']) / test['q_actual'].clip(lower=1)).median()
rmse = np.sqrt(((test['pred_q'] - test['q_actual'])**2).mean())

log('## 8. Holdout (last 20 weeks)')
log(f'- train rows: {len(train):,}, test rows: {len(test):,}')
log(f'- median APE: {mape*100:.1f}%')
log(f'- RMSE: {rmse:.2f} units')

fig, ax = plt.subplots(figsize=(6, 6))
sample = test.sample(min(5000, len(test)), random_state=0)
ax.scatter(sample['q_actual'], sample['pred_q'], s=4, alpha=0.3)
lim = max(sample['q_actual'].quantile(0.99), sample['pred_q'].quantile(0.99))
ax.plot([0, lim], [0, lim], 'k--', lw=1)
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.set_xlabel('actual quantity'); ax.set_ylabel('predicted quantity')
ax.set_title(f'Holdout: predicted vs actual (median APE {mape*100:.1f}%)')
fig.tight_layout(); fig.savefig(FIGURES / 'demand_holdout_scatter.png', dpi=120)
plt.show()


## 9. Elasticity table + interpretation


In [ ]:
def row_for(res, label):
    return {
        'model': label,
        'n_obs': int(res.nobs),
        'R2_within': round(float(res.rsquared), 4),
        'beta_own_price':   round(float(res.params['log_p']), 4),
        'se_own_price':     round(float(res.std_errors['log_p']), 4),
        'beta_cross_price': round(float(res.params.get('log_comp_price', np.nan)), 4),
        'se_cross_price':   round(float(res.std_errors.get('log_comp_price', np.nan)), 4),
        'beta_promo':       round(float(res.params['promo_any_int']), 4),
        'se_promo':         round(float(res.std_errors['promo_any_int']), 4),
        'promo_uplift_%':   round((np.exp(res.params['promo_any_int']) - 1) * 100, 2),
    }

coef_table = pd.DataFrame([
    row_for(res1,    'baseline_own_only'),
    row_for(res2,    'baseline_with_cross'),
    row_for(res_upc, 'robustness_upc'),
])
coef_table['smearing'] = [np.nan, smearing, smearing_upc]

out_csv = PROCESSED / 'model_coefficients.csv'
coef_table.to_csv(out_csv, index=False)
log('## 9. Coefficient table')
for _, r in coef_table.iterrows():
    log(f'- {r["model"]:>22s}: own β={r["beta_own_price"]:+.3f}, '
        f'cross β={r["beta_cross_price"]:+.3f}, promo β={r["beta_promo"]:+.3f} '
        f'(uplift {r["promo_uplift_%"]:+.1f}%), R²w={r["R2_within"]}')
log(f'- saved: {out_csv.relative_to(PROJECT_ROOT)}')

# coefficient bar chart
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(3)
ax.bar(x - 0.25, coef_table['beta_own_price'],   0.25, label='own-price elasticity')
ax.bar(x,        coef_table['beta_cross_price'], 0.25, label='cross-price elasticity')
ax.bar(x + 0.25, coef_table['beta_promo'],       0.25, label='promo semi-elasticity')
ax.set_xticks(x); ax.set_xticklabels(coef_table['model'], rotation=10)
ax.axhline(0, color='k', lw=0.5)
ax.legend(loc='best', fontsize=9)
ax.set_title('Demand coefficient comparison')
fig.tight_layout(); fig.savefig(FIGURES / 'demand_coefficients.png', dpi=120)
plt.show()


## 10. Save artifacts + model limitations


In [ ]:
# save the modeling dataset used for baseline fit (minus intermediate totals)
keep = ['brand_final','size_oz_rounded','STORE','WEEK','week_start_date',
        'quantity','unit_price','unit_cost','promo_any','promo_share',
        'log_q','log_p','log_comp_price','promo_any_int',
        'brand_size_store']
mdl[keep].to_parquet(PROCESSED / 'demand_modeling_dataset.parquet',
                     index=False, compression='snappy')

limitations = [
    '',
    '## Limitations (to disclose in memo)',
    '',
    '1. **No IV**: own-price is treated as exogenous conditional on brand-size-store and week FE. '
    'Remaining endogeneity (unobserved within-cell shocks correlated with price) biases β_own toward 0. '
    'Next notebook adds Hausman IV (same-brand prices in other stores) and cost-shock IV.',
    '2. **AAC ≠ marginal cost**: PROFIT-based unit_cost drags behind true replacement cost (see rawData/README §3.4). '
    'Margin numbers are directionally useful but not a strict marginal-cost measure.',
    '3. **Competitor price index is simple quantity-weighted mean**: not a full demand system. '
    'Cross-elasticity here is a reduced-form substitution signal, not a utility-based Nevo-style estimate.',
    '4. **Holdout is the last 20 weeks only**: pure forecast test, not out-of-sample cross-validation. '
    'Store-week rollout of new UPCs is censored.',
    '5. **Unknown SALE code G** (~0.15% of aggregated rows) is bucketed as promo but its exact mechanism is undocumented.',
    '6. **brand_confidence=low UPCs excluded** (31 UPCs, including the 4 Nabisco/Post conflict UPCs) — robustness panel keeps them.',
]
for line in limitations:
    log(line)

(REPORTS / 'demand_model_summary.md').write_text('\n'.join(summary_lines), encoding='utf-8')
log(f'')
log(f'- saved: reports/demand_model_summary.md')
log(f'- saved: data/processed/demand_modeling_dataset.parquet')


---

## 下一步：`04_counterfactual.ipynb`

- Price elasticities + unit_cost + smearing → revenue/profit under counterfactual prices
- Holiday / competitor-price interactions
- IV-robust estimates (Hausman across-store, cost-shock)
- Memo: top-10 SKU price actions ranked by profit uplift
